# 01 - Join & Clean Daily Covers Data

**Goal:** combine four separate exports (POS, reservations, weather, calendar/events) into one clean, daily-level table with no gaps, documented outliers, and consistent columns — ready for exploratory analysis and modeling.

This uses **synthetic data** built to behave like a real restaurant (weekday patterns, seasonality, holidays, weather effects) so you can learn the pipeline now and swap in real exports later without changing any code below. It now spans close to **2 years** (Aug 2024-Aug 2026), extended from an initial 12 months specifically so later notebooks can check whether a seasonal pattern repeats year over year, rather than relying on a single cycle.

We'll work through it in the order you'd do it for real:
1. Load each raw file and standardize its date/column format
2. Join everything onto one row-per-date table
3. Find missing days
4. Find outliers and decide what to do with each one
5. Build the final feature flags and save the cleaned dataset


## Step 1: Load each source and standardize

Each export has its own date format and column names — this is normal. Standardize each one *before* joining, not after.

In [ ]:
import pandas as pd
import numpy as np

pos = pd.read_csv("../data/raw/pos_covers_export.csv")
pos = pos.rename(columns={"Business Date": "date", "Total Covers": "covers"})
pos["date"] = pd.to_datetime(pos["date"], format="%m/%d/%Y")
pos.head()

**Check for duplicate dates.** POS systems occasionally export a day twice (a re-run, a correction that didn't overwrite). Never assume the first row is right — look at both.

In [ ]:
dupes = pos[pos.duplicated(subset="date", keep=False)]
print(dupes)

# Keep the first entry here; in a real case you'd check the POS system directly
# to see which value is correct rather than guessing.
pos = pos.sort_values("date").drop_duplicates(subset="date", keep="first")

In [ ]:
resv = pd.read_csv("../data/raw/reservations_export.csv")
resv = resv.rename(columns={"covers_reserved": "reservations_covers"})
resv["date"] = pd.to_datetime(resv["date"])

weather = pd.read_csv("../data/raw/weather_export.csv")
weather["date"] = pd.to_datetime(weather["date"])

cal = pd.read_csv("../data/raw/calendar_events.csv")
cal["date"] = pd.to_datetime(cal["date"])

print(resv.shape, weather.shape, cal.shape)

## Step 2: Join onto a full date spine

**Why a "spine"?** If you just merge the files together, a date that's missing from *every* file simply never appears — you won't notice the gap. Building an explicit list of every calendar date first, then left-joining everything onto it, makes missing days visible instead of silently absent.

In [ ]:
full_range = pd.DataFrame({
    "date": pd.date_range(pos["date"].min(), pos["date"].max(), freq="D")
})
print(f"Spine covers {full_range['date'].min().date()} to {full_range['date'].max().date()}, "
      f"{len(full_range)} days")

df = full_range.merge(pos, on="date", how="left")
df = df.merge(resv, on="date", how="left")
df = df.merge(weather, on="date", how="left")
df = df.merge(cal, on="date", how="left")
df.shape

## Step 3: Find missing days

Different columns will be missing for different reasons, and that changes how you handle them:
- **Missing `covers` (the target)** — can't be invented. Flag it and exclude from model training later.
- **Missing weather** — temperature/precipitation are smooth/continuous, so interpolating from neighboring days is reasonable.
- **Missing reservations** — could mean "no reservations that day" or "export failed." Don't assume 0; flag it instead.

In [ ]:
print("Missing covers (target) days:", df["covers"].isna().sum())
print(df.loc[df["covers"].isna(), "date"].dt.date.tolist())

print("\nMissing weather days:", df["temp_high_f"].isna().sum())
print("Missing reservations days:", df["reservations_covers"].isna().sum())

In [ ]:
# Weather: interpolate (smooth variable, neighboring days are a good estimate)
df[["temp_high_f", "precip_in"]] = df[["temp_high_f", "precip_in"]].interpolate(limit_direction="both")
df["condition"] = df["condition"].ffill()

# Reservations: keep NaN as NaN, add an explicit flag rather than guessing 0
df["reservations_missing"] = df["reservations_covers"].isna().astype(int)

# Target (covers): leave as NaN. We will exclude these rows from model
# training later, but keep them in the table so the date sequence stays intact.
df["covers_missing"] = df["covers"].isna().astype(int)

## Step 4: Find and investigate outliers

A raw threshold on `covers` isn't fair — Saturdays are supposed to be much busier than Tuesdays. Compare each day only against **its own day-of-week**, using a z-score (how many standard deviations from that weekday's average).

In [ ]:
df["day_of_week"] = df["date"].dt.day_name()
df["z_covers"] = df.groupby("day_of_week")["covers"].transform(
    lambda s: (s - s.mean()) / s.std()
)

flagged = df.loc[df["z_covers"].abs() > 2.0,
                  ["date", "day_of_week", "covers", "holiday_name", "event_name", "promo_name", "notes", "z_covers"]]
flagged.sort_values("z_covers", key=abs, ascending=False)

**Read each flagged row before touching it.** Most of these turn out to be *real, explainable demand* — keep them, since `is_holiday`/`is_local_event`/`is_promo` flags will let the model learn from them:

- Private buyout, big holidays (Mother's Day, NYE) — real, explained by a feature. **Keep as-is.**
- Burst pipe / health inspection closures — real, but driven by an operational event with no feature to explain it. **Keep, but the `notes` column documents why**, so a future you (or a model of anomalies) isn't confused.
- The **2026-03-05** spike has no holiday, event, promo, or note attached, and doesn't match the natural size of other Thursdays. That's the signature of a **data-entry error** (e.g. a POS double-count), not real demand.

In [ ]:
# Investigate the unexplained spike specifically
suspect = df[df["date"] == "2026-03-05"]
print(suspect[["date", "day_of_week", "covers", "holiday_name", "event_name", "promo_name", "notes"]])

thursdays = df.loc[df["day_of_week"] == "Thursday", "covers"]
print("\nOther Thursdays (sorted):", sorted(thursdays.dropna().tolist()))

In [ ]:
# No feature explains it and it sits well above every other Thursday.
# Correct it to that weekday's median rather than deleting the row (deleting
# would create a second, harder-to-spot gap in the date sequence).
thu_median = df.loc[df["day_of_week"] == "Thursday", "covers"].median()
df.loc[df["date"] == "2026-03-05", "covers"] = thu_median
df.loc[df["date"] == "2026-03-05", "notes"] = "Corrected: POS double-count identified during cleaning, reset to Thursday median"
print(f"Corrected 2026-03-05 to {thu_median}")

## Step 5: Build final feature flags

Derive the calendar/date features from Step 2 of the schema, and turn the calendar file's text columns into clean binary flags.

In [ ]:
df["is_weekend"] = df["day_of_week"].isin(["Friday", "Saturday", "Sunday"]).astype(int)
df["month"] = df["date"].dt.month
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)

# The POS export doesn't split reservations vs walk-ins directly -- derive it.
# NaN reservations (flagged above) will make walk_in_covers NaN too, which is
# correct: we genuinely don't know the split for those days.
df["walk_in_covers"] = df["covers"] - df["reservations_covers"]
df["walk_in_covers"] = df["walk_in_covers"].clip(lower=0)

df["is_holiday"] = df["holiday_name"].notna().astype(int)
df["is_local_event"] = df["event_name"].notna().astype(int)
df["is_promo"] = df["promo_name"].notna().astype(int)
df["is_closed"] = df["is_closed"].fillna(0).astype(int)

for c in ["holiday_name", "event_name", "promo_name", "notes", "condition"]:
    df[c] = df[c].fillna("")

df[["date", "day_of_week", "is_weekend", "covers", "is_holiday", "holiday_name",
    "is_local_event", "event_name", "is_promo", "promo_name"]].head(10)

## Step 6: Save the cleaned dataset

Reorder columns to match the schema from `covers_data_schema_template.xlsx`, then write out the file the rest of the project will build on.

In [ ]:
final_cols = [
    "date", "day_of_week", "is_weekend", "month", "week_of_year",
    "covers", "covers_missing", "reservations_covers", "reservations_missing", "walk_in_covers",
    "is_holiday", "holiday_name",
    "is_local_event", "event_name",
    "is_promo", "promo_name",
    "temp_high_f", "precip_in", "condition",
    "is_closed", "notes",
]
clean = df[final_cols].sort_values("date").reset_index(drop=True)
clean.to_csv("../data/processed/covers_daily_clean.csv", index=False)
print(clean.shape)
clean.head()

## Recap

- Joined 4 raw exports (different date formats and column names) onto one row-per-date table
- Found and flagged 7 missing target days, 29 missing reservation days, 18 missing weather days, and 2 duplicate POS rows (spanning close to 2 years of combined history now)
- Investigated every statistical outlier before touching it — kept the real, explainable ones (holidays, events, buyouts) and corrected one unexplained data-entry error
- Saved `data/processed/covers_daily_clean.csv`, matching the schema in `covers_data_schema_template.xlsx`

**Next step:** exploratory analysis — trend, weekday patterns, and the effect of weather/holidays/promos on covers.